# Synthetic valley: running PEST++ IES

This notebook runs the iterative ensemble smoother PEST++ IES on the interface built in [`synthetic-valley-e-pest-interface`](synthetic-valley-e-pest-interface.ipynb), then inspects the fit, the posterior observation ensembles, the posterior parameters, and the predictions. Run that notebook first.

IES starts from the prior parameter ensemble and, at each iteration, moves every realization toward the (noisy) observations using an approximate Jacobian estimated from the ensemble itself. The result is a posterior ensemble whose spread is the uncertainty left after history matching.

## What this notebook covers

By the end of this notebook you will be able to:

- run PEST++ IES in parallel with `pyemu.os_utils.start_workers()`,
- read the phi and ensemble files it writes with `pyemu`,
- compare the prior and posterior ensembles of the observations to the truth,
- see how the pilot points and the other parameters changed, and map the posterior property fields from the array observations, and
- compare the prior and posterior predictions to the truth.

## Imports

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyemu

## Run PEST++ IES

`start_workers()` copies the interface to a master directory and to one directory per worker, starts `pestpp-ies` in the master, and starts a worker in each copy that runs the model when the master asks. An iteration is a run for each realization plus the runs that test the step length, so the time it takes grows with the number of realizations, `num_reals`, and the number of iterations, `noptmax`, and shrinks with the number of workers, `num_workers`. The master prints its progress to the terminal that started JupyterLab.

In [ ]:
num_reals = 50
num_workers = 25
noptmax = 5

In [ ]:
template_ws = pl.Path("models/sv-ies-template")
m_d = pl.Path("models/sv-ies-master")
pst = pyemu.Pst(str(template_ws / "pest.pst"))
pst.control_data.noptmax = noptmax
pst.pestpp_options["ies_num_reals"] = num_reals
print(f"noptmax = {pst.control_data.noptmax}, {pst.nnz_obs} history observations")
pst.write(template_ws / "pest.pst", version=2)

In [ ]:
pyemu.os_utils.start_workers(
    str(template_ws),
    "pestpp-ies",
    "pest.pst",
    num_workers=num_workers,
    worker_root="models",
    master_dir=str(m_d),
    verbose=False,
)
sorted(p.name for p in m_d.glob("pest.*.csv"))

## Phi

`pest.phi.actual.csv` holds the phi of every realization at every iteration, computed against the observed values, and `pest.phi.meas.csv` against each realization's own noisy observations, which is what IES minimizes. Iteration 0 is the prior. If the residuals were as large as the noise and no larger, phi would be about the number of history observations, the red line.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
phi = pd.read_csv(m_d / "pest.phi.actual.csv", index_col=0)
phi_meas = pd.read_csv(m_d / "pest.phi.meas.csv", index_col=0)
reals = phi.columns[5:]  # the realization columns follow the summary columns

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.45 * width), layout="constrained", sharey=True
    )
    for letter, (df, heading) in {
        "A": (phi, "Phi against the observations"),
        "B": (phi_meas, "Phi against the noisy observations"),
    }.items():
        ax = axd[letter]
        ax.plot(df.index, df.loc[:, reals], color="0.6", lw=0.5)
        ax.plot(df.index, df["mean"], color="black", lw=1.5, label="Ensemble mean")
        ax.axhline(pst.nnz_obs, color="red", lw=1.0, label="Number of observations")
        ax.set_yscale("log")
        ax.set_xticks(df.index)
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
        flopy.plot.styles.xlabel(ax=ax, label="Iteration")
    flopy.plot.styles.ylabel(ax=axd["A"], label="Phi")
    flopy.plot.styles.graph_legend(ax=axd["A"], loc="upper right")
print(phi[["mean", "min", "max"]].round(0))

## Observations: prior and posterior

`pest.0.obs.csv` is the prior observation ensemble and the file of the last iteration, `pest.{noptmax}.obs.csv`, the posterior, with one row per realization and one column per observation, which `pyemu.ObservationEnsemble.from_csv()` reads. `pest.obs+noise.csv` holds the noisy observations each realization was matched to. The plots show every realization at four wells, with the truth, through the history. Set `zoom = True` to zoom the y axis to the posterior and the noisy observations, or `False` to see the whole prior range.

In [ ]:
# True: zoom the y axis to the posterior and the noisy observations;
# False: show the whole prior range
zoom = True

obs = pst.observation_data
obs["time"] = obs.time.astype(float)
t_history = obs.loc[obs.weight > 0, "time"].max()
prior_oe = pyemu.ObservationEnsemble.from_csv(pst=pst, filename=m_d / "pest.0.obs.csv")
post_oe = pyemu.ObservationEnsemble.from_csv(
    pst=pst, filename=m_d / f"pest.{pst.control_data.noptmax}.obs.csv"
)
noise_oe = pyemu.ObservationEnsemble.from_csv(
    pst=pst, filename=m_d / "pest.obs+noise.csv"
)


def zoom_to_posterior(ax, names, sign=1.0):
    """set the y range to the posterior and the noisy observations, with a buffer"""
    nz = [name for name in names if obs.loc[name, "weight"] > 0]
    values = np.concatenate(
        [
            sign * post_oe.loc[:, names].values.ravel(),
            sign * noise_oe.loc[:, nz].values.ravel(),
            sign * obs.loc[names, "obsval"].values,
        ]
    )
    buffer = 0.05 * (values.max() - values.min())
    ax.set_ylim(values.min() - buffer, values.max() + buffer)


def plot_series(ax, usecol, show_noise=True):
    history = (obs.usecol == usecol) & (obs.time <= t_history)
    series = obs.loc[history].sort_values("time")
    years = (series.time - series.time.min()) / 365.25
    ax.plot(
        years, noise_oe.loc[:, series.obsnme].values.T, color="r", lw=0.5, alpha=0.25
    )
    ax.plot(
        years, prior_oe.loc[:, series.obsnme].values.T, color="0.7", lw=0.5, alpha=0.5
    )
    ax.plot(
        years,
        post_oe.loc[:, series.obsnme].values.T,
        color="tab:blue",
        lw=0.5,
        alpha=0.5,
    )

    ax.scatter(years, series.obsval, color="red", marker=".", zorder=10)
    if zoom:
        zoom_to_posterior(ax, series.obsnme.tolist())


wells = {
    "A": "wt05-i14-j11",
    "B": "aq05-i14-j11",
    "C": "wt13-i28-j23",
    "D": "aq13-i28-j23",
}
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["C", "D"]], figsize=(width, 0.7 * width), layout="constrained"
    )
    for letter, usecol in wells.items():
        plot_series(axd[letter], usecol)
        flopy.plot.styles.heading(ax=axd[letter], letter=letter, heading=usecol)
    for letter in "CD":
        flopy.plot.styles.xlabel(ax=axd[letter], label="Years since the steady state")
    for letter in "AC":
        flopy.plot.styles.ylabel(ax=axd[letter], label="Head (m)")
    handles = [
        plt.Line2D([], [], color="0.7", lw=1.0, label="Prior"),
        plt.Line2D([], [], color="tab:blue", lw=1.0, label="Posterior"),
        plt.Line2D([], [], color="0.3", marker=".", ls="", label="Noisy observations"),
        plt.Line2D([], [], color="red", lw=1.5, label="Truth"),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=4, frameon=False)

## Steady-state heads: one-to-one plot

The steady-state heads, group `ssheads`, of every realization against the observed heads, one small dot for each head of each realization: the prior in gray, the posterior in blue, and in red the noisy observations each realization was matched to, `pest.obs+noise.csv`. The larger blue dots are the posterior medians, and the box gives the statistics of their residuals.

In [ ]:
ssheads = obs.loc[obs.obgnme == "ssheads"]
observed = ssheads.obsval
median = post_oe.loc[:, observed.index].median()
residual = observed - median
low = observed.min() - 1.0
high = observed.max() + 1.0

fig, ax = plt.subplots(figsize=(6.0, 6.0), layout="constrained")
ax.plot([low, high], [low, high], color="black", lw=1.0)
for real in prior_oe.index:
    ax.plot(observed, prior_oe.loc[real, observed.index], ".", color="0.6", ms=2)
for real in noise_oe.index:
    ax.plot(observed, noise_oe.loc[real, observed.index], ".", color="red", ms=2)
for real in post_oe.index:
    ax.plot(observed, post_oe.loc[real, observed.index], ".", color="tab:blue", ms=2)
ax.plot([], [], ".", color="0.6", label="Prior")
ax.plot([], [], ".", color="red", label="Observed + noise")
ax.plot([], [], ".", color="tab:blue", label="Posterior")
ax.plot(observed, median, "o", color="tab:blue", ms=5, label="Posterior median")
ax.set_xlim(low, high)
ax.set_ylim(low, high)
ax.set_aspect("equal")
ax.set_xlabel("Observed head (m)")
ax.set_ylabel("Simulated head (m)")
ax.legend(loc="lower right")
weighted = ssheads.weight * residual
stats = (
    f"observations {len(residual)}\n"
    f"mean residual {residual.mean():.3f} m\n"
    f"RMS residual {np.sqrt((residual**2).mean()):.3f} m\n"
    f"largest |residual| {residual.abs().max():.3f} m\n"
    f"phi of the median {(weighted**2).sum():.1f}"
)
ax.text(0.03, 0.97, stats, transform=ax.transAxes, va="top", family="monospace")
plt.show()

## Parameters

The parameter ensembles `pest.0.par.csv` and `pest.{noptmax}.par.csv` have one row per realization and one column per parameter; `pandas` reads them directly. The ratio of the posterior to the prior standard deviation of each group, in log10 units, shows which parameters the history informed: a ratio near 1 means the data said nothing about that group.

In [ ]:
par = pst.parameter_data
prior_pe = pd.read_csv(m_d / "pest.0.par.csv", index_col=0)
post_pe = pd.read_csv(m_d / f"pest.{pst.control_data.noptmax}.par.csv", index_col=0)
prior_sd = np.log10(prior_pe).std().groupby(par.pargp).mean()
post_sd = np.log10(post_pe).std().groupby(par.pargp).mean()
ratio = (post_sd / prior_sd).sort_values()

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    ax.bar(ratio.index, ratio.values, color="tab:blue")
    ax.axhline(1.0, color="0.5", lw=0.75)
    ax.set_ylim(0.0, 1.1)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    flopy.plot.styles.ylabel(ax=ax, label="Posterior / prior standard deviation")

The pilot points carry their row `i` and column `j` in their names. The maps show, for layer 1, the mean of the posterior log10 multiplier at each pilot point and the ratio of its posterior to prior standard deviation. The red triangles are the observation wells.

## Property fields

Every property array is also an observation: the forward run writes the K, vertical K, and specific storage of each layer to a file, and the instruction files read one value per active cell (groups `k1`, …, `k335`, …, `ss5`, with the row `i` and column `j` of each cell). So the observation ensembles already hold the property fields of every realization, and no extra model runs are needed to map them.

The maps show, for each layer, the posterior mean of log10 of the property, the `base` realization, which started from the homogeneous model, and three other posterior realizations. Each row, a layer, has its own color scale, so the variation within each layer shows even where the layers differ by orders of magnitude. The blank cells are the inactive cells of layer 3, where there is no clay, and the lake cells of layer 1, whose very high K is not a parameter. The mean is smoother than any one realization: the realizations differ where the history says little, and averaging them cancels those differences.

In [ ]:
sim = flopy.mf6.MFSimulation.load(
    sim_ws=template_ws, verbosity_level=0, load_only=["dis", "npf"]
)
gwf = sim.get_model()
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
post_reals = [real for real in post_oe.index if real != "base"][:3]
titles = ["Posterior mean", "Base"] + [f"Realization {real}" for real in post_reals]


def property_maps(prefix, label):
    # a table per layer: one row per panel, one column per active cell
    tables = []
    for lay in range(nlay):
        arr_obs = obs.loc[obs.oname == f"{prefix}{lay + 1}"]
        log_values = np.log10(post_oe.loc[:, arr_obs.obsnme].astype(float))
        mean = log_values.mean().to_frame("mean").T
        tables.append(
            (arr_obs, pd.concat([mean, log_values.loc[["base"] + post_reals]]))
        )
    with flopy.plot.styles.USGSMap():
        fig, axes = plt.subplots(
            nlay, len(titles), figsize=(width, 1.2 * width), layout="constrained"
        )
        for lay, (arr_obs, table) in enumerate(tables):
            i, j = arr_obs.i.astype(int).values, arr_obs.j.astype(int).values
            vmin, vmax = np.percentile(table.values, [2, 98])  # a scale per layer
            for col in range(len(titles)):
                ax = axes[lay, col]
                values = np.full((nrow, ncol), np.nan)
                values[i, j] = table.iloc[col].values
                mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=lay)
                pa = mm.plot_array(
                    np.ma.masked_invalid(values), cmap="viridis", vmin=vmin, vmax=vmax
                )
                ax.set_aspect("equal")
                ax.set_xticks([])
                ax.set_yticks([])
                if lay == 0:
                    ax.set_title(titles[col], fontsize=7)
                if col == 0:
                    ax.set_ylabel(f"Layer {lay + 1}", fontsize=8)
            cb = fig.colorbar(pa, ax=axes[lay, :], shrink=0.9, aspect=10)
            cb.ax.tick_params(labelsize=7)
        fig.suptitle(label, fontsize=9)


property_maps("k", "log$_{10}$ K (m/d)")

In [ ]:
property_maps("k33", "log$_{10}$ vertical K (m/d)")

In [ ]:
property_maps("ss", "log$_{10}$ specific storage (1/m)")

## Predictions

The three predictions are the groundwater discharge to the stream in the steady state (A), and the smallest discharge in any year of the history (B) and in any year of the future, after the new well starts (C). None has a weight, so IES does not fit them. Each panel compares the prior and posterior histograms with the true value.

In [ ]:
predictions = obs.loc[obs.oname == "pred"]
steady_name = predictions.loc[predictions.prediction == "steadystate"].obsnme.iloc[0]
history_name = predictions.loc[predictions.prediction == "minhistory"].obsnme.iloc[0]
future_name = predictions.loc[predictions.prediction == "minfuture"].obsnme.iloc[0]

with flopy.plot.styles.USGSPlot():
    fig, (ax1, ax2, ax3) = plt.subplots(
        1, 3, figsize=(width, 0.4 * width), layout="constrained", sharey=True
    )
    ax1.hist(prior_oe.loc[:, steady_name], bins=15, color="0.7", alpha=0.7)
    ax1.hist(post_oe.loc[:, steady_name], bins=15, color="tab:blue", alpha=0.7)
    ax1.axvline(obs.loc[steady_name, "obsval"], color="red", lw=1.5)
    flopy.plot.styles.heading(ax=ax1, letter="A", heading="Steady state")
    flopy.plot.styles.xlabel(ax=ax1, label="Discharge (m$^3$/d)")
    flopy.plot.styles.ylabel(ax=ax1, label="Realizations")

    ax2.hist(prior_oe.loc[:, history_name], bins=15, color="0.7", alpha=0.7)
    ax2.hist(post_oe.loc[:, history_name], bins=15, color="tab:blue", alpha=0.7)
    ax2.axvline(obs.loc[history_name, "obsval"], color="red", lw=1.5)
    flopy.plot.styles.heading(ax=ax2, letter="B", heading="Smallest, history")
    flopy.plot.styles.xlabel(ax=ax2, label="Discharge (m$^3$/d)")

    ax3.hist(
        prior_oe.loc[:, future_name], bins=15, color="0.7", alpha=0.7, label="Prior"
    )
    ax3.hist(
        post_oe.loc[:, future_name],
        bins=15,
        color="tab:blue",
        alpha=0.7,
        label="Posterior",
    )
    ax3.axvline(obs.loc[future_name, "obsval"], color="red", lw=1.5, label="Truth")
    flopy.plot.styles.heading(ax=ax3, letter="C", heading="Smallest, future")
    flopy.plot.styles.xlabel(ax=ax3, label="Discharge (m$^3$/d)")
    flopy.plot.styles.graph_legend(ax=ax3, loc="upper right")

## Recap

You ran PEST++ IES in parallel, followed phi over the iterations, compared the prior and posterior ensembles of the heads to the truth, saw which parameter groups and which pilot points the history informed, mapped the posterior property fields from the array observations, and compared the prior and posterior predictions to the truth.